# Exercise 2.2: Workforce Transformation Readiness — Trajectory Forecast

In this notebook, we build on the classification results from Notebook 1. Where Notebook 1 answered *"Who is not ready today?"*, this notebook answers the forward-looking question:

> **"Will our learning programs close the gap in time to hit the transformation milestones?"**

We use [`SNOWFLAKE.ML.FORECAST`](https://docs.snowflake.com/en/user-guide/ml-functions/forecast) to project monthly learning completions per department over 12 months, compare them against program milestone gates, and combine everything into a single custom data product.

---

### Why Learning Completions?
Learning completion count is the most actionable leading indicator of transformation readiness:
- Directly computable from `LearningEventHistory` (which has months/years of historical data)
- The metric L&D can directly influence through program design and nudges
- Combined with Notebook 1's readiness tiers, it tells the CHRO: *"Even if we know who's not ready, will the programs get them there?"*

### Data Products Used

| Data Product | Package | Role in This Notebook |
|---|---|---|
| `LearningEventHistory` | SAP SuccessFactors Learning | Primary — provides the historical time-series of completions |
| `WorkforcePerson` | SAP SuccessFactors Employee Central | Department dimension (via `assignment` → `jobdetail`) |

We also consume `workforce_readiness_scores` from **Notebook 1** when building the final combined data product in Step 6.

### No Pre-loaded Datasets
- **Time-series:** computed directly from `LearningEventHistory` × `WorkforcePerson`
- **Milestones:** created inline in this notebook (a small reference table)

In [ ]:
USE WAREHOUSE SYSTEM$STREAMLIT_NOTEBOOK_WH;
USE DATABASE TECHED_DA261;

## Step 1 — Build the Monthly Time-Series from BDC Data

We compute **monthly learning completions per department** directly from the BDC data products. This table is the foundation for everything that follows — the forecast model trains on this history.

**How it works:**
1. Join `LearningEventHistory` with `WorkforcePerson` (via `assignment` → `jobdetail`) to get each employee's department
2. Truncate `CompletionDateTime` to month
3. Aggregate: count of completions, distinct active learners, and total credit hours per department per month
4. Filter out months with fewer than 3 completions (noise reduction)

> **Why completion count, not completion rate?** All events in `LearningEventHistory` are completed by definition — the rate would always be 100%. The *count* of completions per month captures actual learning activity volume, which is what we want to forecast.

After building this, we run three exploration queries:
- **History depth check** — how many months of data per department? ML.FORECAST needs sufficient history.
- **Full preview** — scan the time-series to spot trends or gaps.
- **Line chart** — visual inspection of the top 10 departments' completion trends over time.


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_2
-- Monthly learning completions per department
-- Uses completion COUNT (not rate) since all events are completed;
-- rate would always be 100%
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
WITH emp_dept AS (
    SELECT DISTINCT p."externalId" AS employee_id, j."department" AS department
    FROM WORKFORCEPERSON_V1."workforceperson"."person" p
    JOIN WORKFORCEPERSON_V1."workforceperson"."assignment" a ON p."id" = a."personId"
    JOIN WORKFORCEPERSON_V1."workforceperson"."jobdetail" j ON a."id" = j."_id"
    WHERE j."department" IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY p."externalId" ORDER BY j."startDate" DESC) = 1
)
SELECT
    ed.department,
    DATE_TRUNC('month', leh."CompletionDateTime")            AS month_date,
    COUNT(*)                                                  AS completions,
    COUNT(DISTINCT leh."StudentID")                           AS active_learners,
    COALESCE(SUM(leh."TotalCreditHours"), 0)                  AS total_credit_hours
FROM LEARNINGEVENTHISTORY_V1."learningeventhistory"."learningeventhistory" leh
JOIN emp_dept ed ON leh."StudentID" = ed.employee_id
WHERE leh."CompletionDateTime" IS NOT NULL
GROUP BY ed.department, DATE_TRUNC('month', leh."CompletionDateTime")
HAVING COUNT(*) >= 3;


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_3
-- Preview: how much history do we have per department?
SELECT department,
       MIN(month_date) AS earliest, MAX(month_date) AS latest,
       COUNT(*)        AS num_months
FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series
GROUP BY department
ORDER BY department;


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_4
-- Preview the time-series data
SELECT * FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series
ORDER BY department, month_date;


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
import snowflake.snowpark.context as ctx
import matplotlib.pyplot as plt

session = ctx.get_active_session()
df = session.sql('''
    SELECT department, month_date, completions, active_learners
    FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series
    ORDER BY department, month_date
''').to_pandas()

top_depts = df.groupby('DEPARTMENT')['COMPLETIONS'].sum().nlargest(10).index
df_top = df[df['DEPARTMENT'].isin(top_depts)]

fig, ax = plt.subplots(figsize=(14, 6))
for dept in df_top['DEPARTMENT'].unique():
    d = df_top[df_top['DEPARTMENT'] == dept]
    ax.plot(d['MONTH_DATE'], d['COMPLETIONS'], marker='o', markersize=4, label=str(dept))

ax.set_xlabel('Month')
ax.set_ylabel('Completions')
ax.set_title('Monthly Learning Completions by Department (Top 10, Historical)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## Step 2 — Train the Forecast Model

[`SNOWFLAKE.ML.FORECAST`](https://docs.snowflake.com/en/user-guide/ml-functions/forecast) trains a time-series model — and the key feature is `SERIES_COLNAME`:

| Parameter | Value | What It Does |
|---|---|---|
| `INPUT_DATA` | The monthly time-series view | Historical completions per dept per month |
| `SERIES_COLNAME` | `DEPARTMENT` | Trains **one independent model per department** automatically |
| `TIMESTAMP_COLNAME` | `MONTH_DATE` | The time axis |
| `TARGET_COLNAME` | `COMPLETIONS` | What we're forecasting |

**What makes this powerful:**
- **No labeled data needed** — unlike classification, the model learns purely from the historical pattern
- **One model per department** — each department gets its own tailored forecast, not a one-size-fits-all prediction
- **Pure SQL** — one `CREATE ... FORECAST` statement, no Python, no library installs
- **Only 3 columns** — we deliberately exclude exogenous features so the `FORECAST()` call doesn't require future data inputs


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_5
-- Train forecast model with only the 3 required columns (no exogenous features)
-- so FORECAST() can be called without providing future exogenous data
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST TECHED_DA261.<SCHEMANAME>.learning_trajectory_model(
    INPUT_DATA        => SYSTEM$QUERY_REFERENCE(
        'SELECT DEPARTMENT, MONTH_DATE, COMPLETIONS FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series'
    ),
    SERIES_COLNAME    => 'DEPARTMENT',
    TIMESTAMP_COLNAME => 'MONTH_DATE',
    TARGET_COLNAME    => 'COMPLETIONS'
);

## Step 3 — Generate the 12-Month Forecast

We call the trained model to project **12 months ahead** with a **90% prediction interval**.

The output contains one row per department per forecast month:

| Column | Description |
|---|---|
| `series` | Department ID |
| `ts` | Forecast month (timestamp) |
| `forecast` | Predicted completion count (point estimate) |
| `lower_bound` | 90% CI lower bound — pessimistic scenario |
| `upper_bound` | 90% CI upper bound — optimistic scenario |

The prediction interval is critical for business decision-making: if the *upper bound* is still below the milestone target, even the optimistic scenario won't make it — that department needs intervention now.

> **Technical note:** `FORECAST()` returns results via query ID, so we immediately persist them to `bt_learning_forecast_raw` using `RESULT_SCAN(LAST_QUERY_ID())`.


✍️ `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_6
-- Generate 12-month forecast with 90% prediction interval (upper/lower bounds)
-- RESULT_SCAN captures the output since FORECAST returns results via query ID
CALL TECHED_DA261.<SCHEMANAME>.learning_trajectory_model!FORECAST(
    FORECASTING_PERIODS => 12,
    CONFIG_OBJECT       => {'prediction_interval': 0.90}
);

-- Persist forecast results to a table for downstream joins and ICE mirroring
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.learning_forecast_raw
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX.

In [ ]:
%%sql -r dataframe_7
-- Preview the forecast output
SELECT series     AS department,
       ts         AS forecast_month,
       ROUND(forecast, 1)    AS projected_pct,
       ROUND(lower_bound, 1) AS lower_90,
       ROUND(upper_bound, 1) AS upper_90
FROM  TECHED_DA261.<SCHEMANAME>.learning_forecast_raw
ORDER BY department, forecast_month;

## Step 4 — Create Transformation Milestones & Compare

The transformation program has **milestone gates** — target completion counts each department must hit by specific dates. We create this reference table inline (in production, these would come from the PMO system).

### Milestone Gates

| Milestone | Target Date | Required Completions |
|---|---|---|
| Phase 1 Gate | December 2027 | 50 completions/month |
| Phase 2 Gate | December 2029 | 80 completions/month |

We then compare each department's forecast against these gates. The comparison produces a **milestone status** for every department:

| Status | Logic | What It Means |
|---|---|---|
| `ON_TRACK` | Point forecast ≥ target | Projected to hit the milestone |
| `MARGINAL` | Lower bound ≥ target | On track, but the pessimistic scenario still passes |
| `POSSIBLE_IF_ACCELERATED` | Upper bound ≥ target | Could make it if learning programs are accelerated |
| `WILL_MISS` | Even upper bound < target | Will not hit the milestone — needs intervention |

> **Note:** We only create milestones for the top 6 departments (by historical completions) that also have forecast data. This keeps the lab focused and avoids sparse comparisons.


✍️  Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_8
-- Build milestones for top 6 departments that have both history and forecast
-- Milestone dates aligned to December (the forecast cadence)
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.transformation_milestones
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
WITH top_depts AS (
    SELECT h.DEPARTMENT AS department
    FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series h
    INNER JOIN (SELECT DISTINCT series FROM TECHED_DA261.<SCHEMANAME>.learning_forecast_raw) f
        ON h.DEPARTMENT = f.series
    GROUP BY 1
    ORDER BY SUM(h.COMPLETIONS) DESC
    LIMIT 6
)
SELECT td.department, m.milestone_date, m.milestone_name, m.required_completions
FROM top_depts td
CROSS JOIN (
    SELECT '2027-12-01'::TIMESTAMP_NTZ AS milestone_date, 'Phase 1 Gate' AS milestone_name, 50 AS required_completions
    UNION ALL
    SELECT '2029-12-01'::TIMESTAMP_NTZ, 'Phase 2 Gate', 80
) m;


✍️  Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
%%sql -r dataframe_9
-- Compare forecast vs milestones (nearest forecast month to each milestone)
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.forecast_vs_milestones
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
AS
SELECT
    f.series                                              AS department,
    f.ts                                                  AS forecast_month,
    ROUND(f.forecast, 1)                                  AS projected_completions,
    ROUND(f.lower_bound, 1)                               AS lower_90,
    ROUND(f.upper_bound, 1)                               AS upper_90,
    m.milestone_name,
    m.required_completions,
    CASE
        WHEN f.forecast    >= m.required_completions THEN 'ON_TRACK'
        WHEN f.lower_bound >= m.required_completions THEN 'MARGINAL'
        WHEN f.upper_bound >= m.required_completions THEN 'POSSIBLE_IF_ACCELERATED'
        ELSE 'WILL_MISS'
    END                                                   AS milestone_status
FROM TECHED_DA261.<SCHEMANAME>.learning_forecast_raw  f
JOIN TECHED_DA261.<SCHEMANAME>.transformation_milestones  m
    ON f.series = m.department
    AND DATE_TRUNC('month', f.ts) = DATE_TRUNC('month', m.milestone_date);

## Step 5 — Explore the Forecast Results

Three exploration outputs:

1. **Milestone status summary** — every department × milestone combination, sorted by urgency (WILL_MISS first, ON_TRACK last). This is the table a CHRO would scan first.

2. **Gap analysis** — for departments that are ON_TRACK, how much headroom do they have? (Difference between required and projected completions.)

3. **Forecast visualization** — 2×3 panel chart for the top 6 departments showing:
   - **Solid line:** historical actuals
   - **Dashed line:** 12-month forecast
   - **Shaded band:** 90% confidence interval
   - **Dotted horizontal lines:** milestone gates, color-coded by status

> **What to look for:**
> - Departments where the forecast line is **below** the milestone gate = intervention needed
> - Wide confidence bands = high uncertainty — the model isn't confident about this department's trajectory
> - Departments where actuals are trending **upward** but forecast is flat = possible seasonal pattern the model captured


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX




In [ ]:
%%sql -r dataframe_10
-- Milestone status summary
SELECT department, milestone_name, required_completions,
       projected_completions, milestone_status
FROM TECHED_DA261.<SCHEMANAME>.forecast_vs_milestones
ORDER BY
    CASE milestone_status
        WHEN 'WILL_MISS'                THEN 1
        WHEN 'POSSIBLE_IF_ACCELERATED'  THEN 2
        WHEN 'MARGINAL'                 THEN 3
        WHEN 'ON_TRACK'                 THEN 4
    END;


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX



In [ ]:
%%sql -r dataframe_11
-- Gap analysis: how far off are WILL_MISS departments?
SELECT department, milestone_name, required_completions, projected_completions,
       ROUND(required_completions - projected_completions, 1) AS gap_completions
FROM TECHED_DA261.<SCHEMANAME>.forecast_vs_milestones
WHERE  milestone_status = 'ON_TRACK'
ORDER BY gap_completions DESC;


✍️  Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
import snowflake.snowpark.context as ctx
import matplotlib.pyplot as plt
import pandas as pd

session = ctx.get_active_session()

hist = session.sql('''
    SELECT department, month_date, completions
    FROM TECHED_DA261.<SCHEMANAME>.vw_learning_monthly_series
    ORDER BY department, month_date
''').to_pandas()

fcast = session.sql('''
    SELECT series AS department, ts AS month_date,
           ROUND(forecast, 1) AS completions,
           ROUND(lower_bound, 1) AS lower_90,
           ROUND(upper_bound, 1) AS upper_90
    FROM TECHED_DA261.<SCHEMANAME>.learning_forecast_raw
    ORDER BY department, month_date
''').to_pandas()

miles = session.sql('SELECT * FROM TECHED_DA261.<SCHEMANAME>.forecast_vs_milestones').to_pandas()

# Normalize types: department to string, dates to datetime
hist['DEPARTMENT'] = hist['DEPARTMENT'].astype(str)
hist['MONTH_DATE'] = pd.to_datetime(hist['MONTH_DATE'])

fcast['DEPARTMENT'] = fcast['DEPARTMENT'].astype(str)
fcast['MONTH_DATE'] = pd.to_datetime(fcast['MONTH_DATE'])

miles['DEPARTMENT'] = miles['DEPARTMENT'].astype(str)

# Pick top 6 departments by total historical completions
top_depts = hist.groupby('DEPARTMENT')['COMPLETIONS'].sum().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, dept in zip(axes.flat, top_depts):
    h = hist[hist['DEPARTMENT'] == dept].sort_values('MONTH_DATE')
    f = fcast[fcast['DEPARTMENT'] == dept].sort_values('MONTH_DATE')
    m = miles[miles['DEPARTMENT'] == dept]

    if not h.empty:
        ax.plot(h['MONTH_DATE'], h['COMPLETIONS'], 'o-', color='#354A5E',
                markersize=4, label='Actual')
    if not f.empty:
        ax.plot(f['MONTH_DATE'], f['COMPLETIONS'], 's--', color='#0070F2',
                markersize=3, label='Forecast')
        ax.fill_between(f['MONTH_DATE'], f['LOWER_90'], f['UPPER_90'],
                        alpha=0.2, color='#0070F2', label='90% CI')

    for _, row in m.iterrows():
        color_map = {'ON_TRACK': '#107E3E', 'MARGINAL': '#F0AB00',
                     'POSSIBLE_IF_ACCELERATED': '#E76500', 'WILL_MISS': '#BB0000'}
        ax.axhline(y=row['REQUIRED_COMPLETIONS'], color=color_map.get(row['MILESTONE_STATUS'], '#999'),
                   linestyle=':', alpha=0.7)

    ax.set_title(f'Dept {dept}', fontweight='bold')
    ax.set_ylabel('Completions')
    ax.tick_params(axis='x', rotation=45)

axes[0][0].legend(fontsize=8)
fig.suptitle('Learning Completion Trajectory — Forecast vs Milestones', fontsize=14)
plt.tight_layout()
plt.show()

## Step 6 — Persist the tables

This is where Notebooks 1 and 2 come together. We combine:
- **Employee-level readiness tiers** (from Notebook 1's `workforce_readiness_scores`)
- **Department-level forecast context** (from this notebook's milestone comparison)

into a single table: `workforce_transformation_velocity`

### Output Schema

| Column | Source | Description |
|---|---|---|
| `employee_id` | NB1 | Employee identifier |
| `department` | NB1 (via WorkforcePerson) | Department ID |
| `job_title` | NB1 (via WorkforcePerson) | Job title |
| `readiness_tier` | NB1 ML.CLASSIFICATION | READY_NOW / FAST_TRACK_3MO / NEEDS_COACHING / NEEDS_PROGRAM |
| `readiness_velocity_index` | NB1 ML.CLASSIFICATION | Weighted probability score (0–100) |
| `dept_milestone_status` | NB2 ML.FORECAST | ON_TRACK / MARGINAL / POSSIBLE_IF_ACCELERATED / WILL_MISS |
| `dept_projected_completions_12mo` | NB2 ML.FORECAST | Highest projected monthly completions |

The three code cells below:
1. **Create a summary table** — aggregates forecast vs. milestones to one row per department (worst-case status)
2. **Create the table** — defines the Iceberg V3 schema
3. **Populate it** — joins NB1 scores with NB2 department forecast summary

After populating, we verify the cross-tabulation of `readiness_tier × dept_milestone_status` to confirm the data looks right.



✍️  Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX.


In [ ]:
-- Department-level forecast summary view
-- Aggregates forecast vs milestones to one row per department:
--   dept_milestone_status: worst-case status across all milestones (WILL_MISS > ON_TRACK)
--   dept_projected_completions_12mo: highest projected completion count
CREATE OR REPLACE VIEW TECHED_DA261.<SCHEMANAME>.vw_dept_forecast_summary AS
SELECT
    department,
    MAX(CASE
        WHEN milestone_status = 'WILL_MISS'               THEN 'WILL_MISS'
        WHEN milestone_status = 'POSSIBLE_IF_ACCELERATED'  THEN 'POSSIBLE_IF_ACCELERATED'
        WHEN milestone_status = 'MARGINAL'                 THEN 'MARGINAL'
        ELSE 'ON_TRACK'
    END)                                                   AS dept_milestone_status,
    MAX(projected_completions)                             AS dept_projected_completions_12mo
FROM TECHED_DA261.<SCHEMANAME>.forecast_vs_milestones
GROUP BY department;


✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Create the combined data product table
-- Joins employee-level readiness (from NB1) with department-level forecast context (from NB2)
-- This is the primary table for SAC dashboard consumption
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANME>.workforce_transformation_velocity(
    employee_id              VARCHAR,
    department               NUMBER(38,0),
    job_title                VARCHAR,
    readiness_tier           VARCHAR,
    readiness_velocity_index FLOAT,
    dept_milestone_status    VARCHAR,
    dept_projected_completions_12mo FLOAT
)
ICEBERG_VERSION = 3
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED'
STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'


✍️ 
- Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Populate the combined data product
-- LEFT JOIN ensures all employees appear even if their department has no milestone
-- COALESCE defaults to 'NO_MILESTONE' for departments without forecast data
INSERT INTO TECHED_DA261.<SCHEMANAME>.workforce_transformation_velocity
SELECT
    rs.employee_id, rs.department, rs.job_title,
    rs.readiness_tier,
    rs.readiness_velocity_index,
    COALESCE(df.dept_milestone_status, 'NO_MILESTONE'),
    df.dept_projected_completions_12mo
FROM TECHED_DA261.<SCHEMANAME>.workforce_readiness_scores rs
LEFT JOIN TECHED_DA261.<SCHEMANAME>.vw_dept_forecast_summary df
    ON rs.department = df.department;

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX

In [ ]:
-- Verify the custom data product
SELECT readiness_tier, dept_milestone_status, COUNT(*) AS cnt
FROM  TECHED_DA261.<SCHEMANAME>.workforce_transformation_velocity
GROUP BY 1, 2
ORDER BY 1, 2;

Now that all relevant tables are created, the next exercise focuses on packaging and publishing them as a data product. To do this, each table's column types must be mapped to supported CSN types.

Refer to [SAP Note 3799841](https://me.sap.com/notes/3799841) for the complete list of supported types.

In our case, some tables need to be recreated with compatible types. While this adjustment could have been made during initial table creation, it is handled as a separate step here to make the type-mapping requirement explicit and easier to follow.

✍️ Replace `<SCHEMANAME>` with your username. Eg. TECHED_DA261.TAC12XXXXXXXXX.

In [ ]:
CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.FORECAST_VS_MILESTONES_SAP
        ICEBERG_VERSION = 3
        CATALOG = 'SNOWFLAKE'
        EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
        STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
        ENABLE_ICEBERG_MERGE_ON_READ = FALSE
    AS SELECT
        DEPARTMENT::NUMBER(20,0) AS DEPARTMENT,
        FORECAST_MONTH,
        MILESTONE_NAME,
        MILESTONE_STATUS,
        PROJECTED_COMPLETIONS,
        REQUIRED_COMPLETIONS,
        LOWER_90,
        UPPER_90
    FROM TECHED_DA261.<SCHEMANAME>.FORECAST_VS_MILESTONES;

    CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.LEARNING_FORECAST_RAW_SAP
        ICEBERG_VERSION = 3
        CATALOG = 'SNOWFLAKE'
        EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
        STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
        ENABLE_ICEBERG_MERGE_ON_READ = FALSE
    AS SELECT
        SERIES::NUMBER(20,0) AS SERIES,
        TS,
        FORECAST,
        LOWER_BOUND,
        UPPER_BOUND
    FROM TECHED_DA261.<SCHEMANAME>.LEARNING_FORECAST_RAW;

    CREATE OR REPLACE ICEBERG TABLE TECHED_DA261.<SCHEMANAME>.WORKFORCE_READINESS_SCORES_SAP
        ICEBERG_VERSION = 3
        CATALOG = 'SNOWFLAKE'
        EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
        STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE'
        ENABLE_ICEBERG_MERGE_ON_READ = FALSE
    AS SELECT
        EMPLOYEE_ID,
        DEPARTMENT,
        JOB_TITLE,
        READINESS_TIER,
        P_READY_NOW,
        P_FAST_TRACK,
        P_NEEDS_COACHING,
        P_NEEDS_PROGRAM,
        READINESS_VELOCITY_INDEX,
        PREDICTION::VARCHAR AS PREDICTION
    FROM TECHED_DA261.<SCHEMANAME>.WORKFORCE_READINESS_SCORES;

    DROP TABLE IF EXISTS TECHED_DA261.<SCHEMANAME>.WORKFORCE_READINESS_SCORES;
    DROP TABLE IF EXISTS TECHED_DA261.<SCHEMANAME>.LEARNING_FORECAST_RAW;
    DROP TABLE IF EXISTS TECHED_DA261.<SCHEMANAME>.FORECAST_VS_MILESTONES;


## Done!

The **Workforce Transformation Velocity Intelligence** table is ready. It combines two Snowflake Cortex ML workflows:

| Workflow | ML Function | Granularity | Output |
|---|---|---|---|
| Classification (Notebook 1) | `ML.CLASSIFICATION` | Per employee | Readiness tier + velocity index |
| Forecast (Notebook 2) | `ML.FORECAST` | Per department | 12-month trajectory + milestone status |

All feature tables and output tables created in this notebook are created as Iceberg tables . However, they can also be created as Dynamic Iceberg tables that auto-refresh within 1 hour.

The syntax to create it is as follows:

CREATE OR REPLACE DYNAMIC ICEBERG TABLE DATABASE.SCHEMA.TABLENAME_1 TARGET_LAG = '1 hour' WAREHOUSE = DEMOWAREHOUSE EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED' CATALOG = 'SNOWFLAKE' ICEBERG_MERGE_ON_READ_BEHAVIOR = 'DISABLED' STORAGE_SERIALIZATION_POLICY = 'COMPATIBLE' AS SELECT COLUMNS from DATABASE.SCHEMA.TABLENAME_2

**A note on table type: Iceberg Tables vs. Dynamic Iceberg Tables:**

In this exercise we create standard Iceberg Tables rather than Dynamic Iceberg Tables. Here's why:

Dynamic Iceberg Tables automatically materialize and refresh their results based on a defined target lag. This means Snowflake continuously runs a background refresh process that consumes compute resources (virtual warehouse credits) to keep the table in sync with its upstream sources. While this is powerful for near-real-time pipelines, it is not required for a TechEd Hands-on.

Proceed to the next exercise 3 to create a share and publish the data product.